In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
import dlt


@dlt.table(
    comment="Bronze delta table for stagging"
)
def bronze_orders():
    source_path="abfss://basab2@basabstore2.dfs.core.windows.net/sales/"
    
    schema = StructType([
    StructField("sale_id", IntegerType(), True),
    StructField("product_id", IntegerType(), True),
    StructField("region", StringType(), True),
    StructField("store_id", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("sales_amount", DoubleType(), True),
    StructField("sale_date", TimestampType(), True)
    ])
    raw_stream = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")  # Pass directly
        .schema(schema)
        .load(source_path)
    )
    bronze_df=raw_stream

    return bronze_df


@dlt.table(
    comment="Products delta table for staging"
)
def products_table():
    product_df = spark.read.option("multiline","true").json("abfss://basab2@basabstore2.dfs.core.windows.net/products/products.json")
    return product_df




@dlt.table(
    comment="Silver delta table for transformations"
)
@dlt.expect("valid_quantity","quantity>0")
@dlt.expect("valid_sales_amount","sales_amount>0")
def silver_orders():
    bronze_df=dlt.read("bronze_orders")
    product_df=dlt.read("products_table")

    #Join Korlam🥰
    silver_df = bronze_df.join(product_df, "product_id", "left")  # join on product_id

    
    #New col : Discount ache ki nei ? (Extra Col)
    silver_df=silver_df.withColumn("has_discount",when( col("discount_rate")>0 ,lit(True) ).otherwise(lit(False)))
    #  New Col : Ekta Unit er Price Koto ?
    silver_df=silver_df.withColumn("unit_price",when( col("quantity")>0 ,col("sales_amount")/col("quantity") ).otherwise(lit(0)))
    # New Col : Kerom Revew hoache : high/medium/low
    silver_df=silver_df.withColumn("revenew_tier",when( col("sales_amount")>1000,"High" ).when(col("sales_amount")>500,"Medium").otherwise("Low")) 

    #New Col : Koto Quantity Load royeche : "Bulk/Medium/Low"
    silver_df=silver_df.withColumn("quantity_load",when(col("quantity")>10,"Bulk" ).when(col("quantity")>5,"Medium").otherwise("Low"))
    #New Col : Extracting Store Number :
    silver_df=silver_df.withColumn("store_number",regexp_extract(col("store_id"), r"Store_(\d+)", 1).cast("int")   )
    #New Col : Filling Nulls 
    silver_df=silver_df.fillna(
        {
            "discount_rate":0,
            "unit_price":0,
            
        }
    )
    # Kokhon transform korchi tar timestamp add : New Col
    silver_df = silver_df.withColumn("_ingest_ts", current_timestamp())
    # Jodi duto Sales transaction er ID same hoi then it's invalid, so drop the duplicates
    silver_df = silver_df.dropDuplicates(["sale_id"])

    #Finaly return the table
    return silver_df



@dlt.table(
    comment="Gold delta table for aggregations"
)
def daily_region_sales():
    silver_df=dlt.read("silver_orders")

    silver_df=silver_df.withColumn("sales_date_only",to_date(col("sale_date")) )
    
    gold_agg1=silver_df.groupBy("region","sales_date_only").agg( sum("sales_amount").alias("total_sales") , count("sale_id").alias("total_orders"))

    return gold_agg1


@dlt.table(
    comment="Gold delta table for agg2"
)
def category_performance():
    silver_df=dlt.read("silver_orders")

    gold_agg2=silver_df.groupBy("category").agg( sum("sales_amount").alias("total_sales") , count("sale_id").alias("total_orders"))

    return gold_agg2



@dlt.table(
    comment="Gold delta table for agg3"
)
def store_performance():
    silver_df=dlt.read("silver_orders")
    gold_agg3=silver_df.groupBy("store_number").agg( sum("sales_amount").alias("total_sales") , count("sale_id").alias("total_orders"))

    return gold_agg3




